# Testes de hipótese: infraestrutura escolar, inclusão digital e IDEB

Este notebook usa a base integrada gerada por `01_exploracao_dados.ipynb`. Execute o notebook 01 primeiro para criar `Dados/Tratados/painel_escolas_sp_2023_2025_com_ideb.csv`.

A unidade de análise é a escola pública paulista presente nos Censos Escolares de 2023 e 2025. Os testes de IDEB usam escolas com valores numéricos nos dois anos; o teste territorial de acesso usa escolas com dados válidos de localização e internet para alunos.

## Perguntas e hipóteses sobre IDEB

Para cada recurso, a pergunta é: entre as escolas que não o tinham em 2023, as que passaram a tê-lo em 2025 apresentaram maior variação do IDEB que as que permaneceram sem ele? A variação é `ΔIDEB = IDEB_2025 − IDEB_2023`.

O grupo de interesse é **0→1** e o grupo de referência é **0→0**. As transições **1→0** e **1→1** não entram na comparação porque as escolas já tinham o recurso no início. Também são excluídos registros sem IDEB ou sem classificação válida do indicador.

Os cinco indicadores são internet para aprendizagem, banda larga, laboratório de informática, biblioteca e internet disponível aos alunos. Para cada comparação, **H₀** afirma que a distribuição de `ΔIDEB` no grupo 0→1 não tende a ser maior que no grupo 0→0; **H₁** afirma que tende a ser maior. O teste de Mann–Whitney é unilateral, com correção para empates e continuidade e aproximação normal para o p-valor. Médias e medianas são descrições complementares. Internet para aprendizagem é a hipótese principal; as demais comparações são exploratórias.

In [1]:
from pathlib import Path
from math import sqrt
from statistics import NormalDist
import numpy as np
import pandas as pd

# Funciona com o notebook aberto na raiz do projeto ou em Notebooks/.
PASTA_PROJETO = Path("..") if Path.cwd().name.lower() in {"notebooks", "notebook"} else Path(".")
caminho_base = PASTA_PROJETO / "Dados" / "Tratados" / "painel_escolas_sp_2023_2025_com_ideb.csv"
if not caminho_base.exists():
    raise FileNotFoundError("Execute primeiro 01_exploracao_dados.ipynb para gerar a base integrada.")

indicadores = {
    "Internet para aprendizagem": "IN_INTERNET_APRENDIZAGEM",
    "Banda larga": "IN_BANDA_LARGA",
    "Laboratório de informática": "IN_LABORATORIO_INFORMATICA",
    "Biblioteca": "IN_BIBLIOTECA",
    "Internet para alunos": "IN_INTERNET_ALUNOS",
}
colunas_ideb = ["VL_OBSERVADO_2023", "VL_OBSERVADO_2025"]
colunas_indicadores = [f"{indicador}_{ano}" for indicador in indicadores.values() for ano in (2023, 2025)]
colunas_fisher = ["IN_INTERNET_ALUNOS_2023", "IN_INTERNET_ALUNOS_2025", "TP_LOCALIZACAO_2023"]
colunas = list(dict.fromkeys(colunas_indicadores + colunas_ideb + colunas_fisher))

print("Lendo somente as colunas dos cinco indicadores, IDEB e localização...", flush=True)
df = pd.read_csv(caminho_base, usecols=colunas, encoding="utf-8-sig")
print(f"Escolas na coorte pareada: {len(df):,}.", flush=True)
for coluna in colunas:
    df[coluna] = pd.to_numeric(df[coluna], errors="coerce")

df_ideb = df.dropna(subset=colunas_ideb).copy()
df_ideb["DELTA_IDEB"] = df_ideb["VL_OBSERVADO_2025"] - df_ideb["VL_OBSERVADO_2023"]
print(f"Escolas com IDEB numérico nos dois anos: {len(df_ideb):,}.", flush=True)

resultados_ideb = []
for nome, indicador in indicadores.items():
    flag_2023, flag_2025 = f"{indicador}_2023", f"{indicador}_2025"
    dados = df_ideb.dropna(subset=[flag_2023, flag_2025]).copy()
    if not dados[[flag_2023, flag_2025]].isin([0, 1]).all().all():
        raise ValueError(f"{indicador} precisa conter somente 0, 1 ou ausente.")

    # Compara quem ganhou o recurso (0→1) com quem permaneceu sem ele (0→0).
    grupo_ganhou = dados.loc[dados[flag_2023].eq(0) & dados[flag_2025].eq(1), "DELTA_IDEB"].to_numpy(dtype=float)
    grupo_sem_recurso = dados.loc[dados[flag_2023].eq(0) & dados[flag_2025].eq(0), "DELTA_IDEB"].to_numpy(dtype=float)
    n_outras_transicoes = int(dados[flag_2023].eq(1).sum())
    n_ganhou, n_sem_recurso = len(grupo_ganhou), len(grupo_sem_recurso)
    if n_ganhou == 0 or n_sem_recurso == 0:
        raise ValueError(f"{nome}: os grupos 0→1 e 0→0 precisam ter escolas.")

    combinados = np.concatenate([grupo_ganhou, grupo_sem_recurso])
    postos = pd.Series(combinados).rank(method="average").to_numpy()
    u = float(postos[:n_ganhou].sum() - n_ganhou * (n_ganhou + 1) / 2)
    contagens_empates = pd.Series(combinados).value_counts().to_numpy()
    termo_empates = sum(int(n) ** 3 - int(n) for n in contagens_empates)
    n_total = n_ganhou + n_sem_recurso
    variancia_u = n_ganhou * n_sem_recurso / 12 * (n_total + 1 - termo_empates / (n_total * (n_total - 1)))
    if variancia_u <= 0:
        raise ValueError(f"{nome}: não foi possível calcular a variância de U.")

    # Aproximação normal unilateral, com correções para empates e continuidade.
    z = (u - n_ganhou * n_sem_recurso / 2 - 0.5) / sqrt(variancia_u)
    p_bruto = 1 - NormalDist().cdf(z)
    efeito = 2 * u / (n_ganhou * n_sem_recurso) - 1
    resultados_ideb.append({
        "Teste": nome, "Método": "Mann–Whitney unilateral",
        "n_0→1": n_ganhou, "n_0→0": n_sem_recurso,
        "n_outras_transições": n_outras_transicoes,
        "n_indicador_ausente": len(df_ideb) - len(dados),
        "média_ΔIDEB_0→1": float(grupo_ganhou.mean()),
        "média_ΔIDEB_0→0": float(grupo_sem_recurso.mean()),
        "mediana_ΔIDEB_0→1": float(np.median(grupo_ganhou)),
        "mediana_ΔIDEB_0→0": float(np.median(grupo_sem_recurso)),
        "U": u, "p_bruto": p_bruto, "efeito_rank_biserial": efeito,
    })

print("\nCinco testes de IDEB calculados. O ajuste de Holm será aplicado junto com o teste territorial na próxima célula.")
for r in resultados_ideb:
    print(f"{r['Teste']}: 0→1 n={r['n_0→1']}, 0→0 n={r['n_0→0']}, médias ΔIDEB={r['média_ΔIDEB_0→1']:.3f}/{r['média_ΔIDEB_0→0']:.3f}, p bruto={r['p_bruto']:.4f}, efeito rank-biserial={r['efeito_rank_biserial']:.4f}")


Lendo somente as colunas dos cinco indicadores, IDEB e localização...
Escolas na coorte pareada: 5,445.
Escolas com IDEB numérico nos dois anos: 4,437.

Cinco testes de IDEB calculados. O ajuste de Holm será aplicado junto com o teste territorial na próxima célula.
Internet para aprendizagem: 0→1 n=67, 0→0 n=27, médias ΔIDEB=0.064/0.015, p bruto=0.4733, efeito rank-biserial=0.0094
Banda larga: 0→1 n=65, 0→0 n=118, médias ΔIDEB=0.100/0.074, p bruto=0.2884, efeito rank-biserial=0.0499
Laboratório de informática: 0→1 n=146, 0→0 n=670, médias ΔIDEB=0.097/0.050, p bruto=0.1604, efeito rank-biserial=0.0523
Biblioteca: 0→1 n=147, 0→0 n=3851, médias ΔIDEB=0.054/0.069, p bruto=0.5211, efeito rank-biserial=-0.0026
Internet para alunos: 0→1 n=320, 0→0 n=460, médias ΔIDEB=0.029/0.050, p bruto=0.7023, efeito rank-biserial=-0.0223


## Hipótese exploratória: expansão da internet para alunos por localização

Entre as escolas sem internet disponível aos alunos em 2023, a proporção que ganhou registro de acesso até 2025 foi diferente entre escolas rurais e urbanas?

- **H₀:** as proporções de escolas que ganharam acesso são iguais nas áreas rurais e urbanas.
- **H₁:** as proporções são diferentes.

A classificação territorial usa a localização de 2023. O teste exato bilateral de Fisher inclui as escolas pareadas com localização e indicador válidos, sem exigir IDEB. O p-valor será ajustado junto com os cinco testes de IDEB pelo método de Holm. A associação não demonstra causalidade.

In [2]:
from math import comb

# Calcula o p-valor bilateral exato pela distribuição hipergeométrica.
def fisher_exato_bilateral(a, b, c, d):
    total = a + b + c + d
    linha_rural = a + b
    coluna_ganhou = a + c
    denominador = comb(total, linha_rural)
    numerador_observado = comb(coluna_ganhou, a) * comb(total - coluna_ganhou, linha_rural - a)
    prob_observada = numerador_observado / denominador
    minimo = max(0, linha_rural - (total - coluna_ganhou))
    maximo = min(linha_rural, coluna_ganhou)
    p = 0.0
    for ganhos_rurais in range(minimo, maximo + 1):
        numerador = comb(coluna_ganhou, ganhos_rurais) * comb(total - coluna_ganhou, linha_rural - ganhos_rurais)
        probabilidade = numerador / denominador
        if probabilidade <= prob_observada * (1 + 1e-12):
            p += probabilidade
    return min(1.0, p)

# Usa todas as escolas pareadas com localização e internet válidas, sem exigir IDEB.
# A população do teste é quem não tinha internet para alunos em 2023.
dados_fisher = df.loc[
    df["IN_INTERNET_ALUNOS_2023"].eq(0)
    & df["IN_INTERNET_ALUNOS_2025"].isin([0, 1])
    & df["TP_LOCALIZACAO_2023"].isin([1, 2])
].copy()
dados_fisher["LOCALIZACAO"] = dados_fisher["TP_LOCALIZACAO_2023"].map({1: "Urbana", 2: "Rural"})
dados_fisher["GANHOU_ACESSO"] = dados_fisher["IN_INTERNET_ALUNOS_2025"].eq(1)
tabela = pd.crosstab(dados_fisher["LOCALIZACAO"], dados_fisher["GANHOU_ACESSO"])
tabela = tabela.reindex(index=["Rural", "Urbana"], columns=[True, False], fill_value=0)

a, b = int(tabela.loc["Rural", True]), int(tabela.loc["Rural", False])
c, d = int(tabela.loc["Urbana", True]), int(tabela.loc["Urbana", False])
n_rural, n_urbana = a + b, c + d
taxa_rural, taxa_urbana = a / n_rural, c / n_urbana
p_fisher = fisher_exato_bilateral(a, b, c, d)
diferenca_pp = (taxa_rural - taxa_urbana) * 100

# Holm step-down aplicado conjuntamente aos cinco testes de IDEB e ao Fisher.
todos_p = [r["p_bruto"] for r in resultados_ideb] + [p_fisher]
ordem = sorted(range(len(todos_p)), key=lambda i: todos_p[i])
p_ajustado_anterior = 0.0
p_holm = [None] * len(todos_p)
for posicao, indice in enumerate(ordem):
    p_candidato = min(1.0, (len(todos_p) - posicao) * todos_p[indice])
    p_ajustado_anterior = max(p_ajustado_anterior, p_candidato)
    p_holm[indice] = p_ajustado_anterior
for i, r in enumerate(resultados_ideb):
    r["p_Holm_6_testes"] = p_holm[i]

print("Tabela de contingência (ganhou / permaneceu sem internet):")
print(tabela.rename(columns={True: "Ganhou", False: "Permaneceu sem internet"}).to_string())
print(f"\nRurais: {a}/{n_rural} ({taxa_rural:.1%}); urbanas: {c}/{n_urbana} ({taxa_urbana:.1%}).")
print(f"Diferença rural - urbana: {diferenca_pp:.1f} pontos percentuais.")
print(f"Fisher exato bilateral: p bruto={p_fisher:.6f}; p-Holm (seis testes)={p_holm[-1]:.6f}.")
print("\nResumo dos seis testes (α = 0,05; ajuste de Holm conjunto):")
for r in resultados_ideb:
    decisao = "rejeita H₀" if r["p_Holm_6_testes"] < 0.05 else "não rejeita H₀"
    print(f"{r['Teste']}: p bruto={r['p_bruto']:.4f}; p-Holm={r['p_Holm_6_testes']:.4f}; {decisao}.")
decisao_fisher = "rejeita H₀" if p_holm[-1] < 0.05 else "não rejeita H₀"
print(f"Internet para alunos: rural vs. urbana: p bruto={p_fisher:.6f}; p-Holm={p_holm[-1]:.6f}; {decisao_fisher}.")


Tabela de contingência (ganhou / permaneceu sem internet):
GANHOU_ACESSO  Ganhou  Permaneceu sem internet
LOCALIZACAO                                   
Rural              40                       32
Urbana            372                      766

Rurais: 40/72 (55.6%); urbanas: 372/1138 (32.7%).
Diferença rural - urbana: 22.9 pontos percentuais.
Fisher exato bilateral: p bruto=0.000158; p-Holm (seis testes)=0.000946.

Resumo dos seis testes (α = 0,05; ajuste de Holm conjunto):
Internet para aprendizagem: p bruto=0.4733; p-Holm=1.0000; não rejeita H₀.
Banda larga: p bruto=0.2884; p-Holm=1.0000; não rejeita H₀.
Laboratório de informática: p bruto=0.1604; p-Holm=0.8021; não rejeita H₀.
Biblioteca: p bruto=0.5211; p-Holm=1.0000; não rejeita H₀.
Internet para alunos: p bruto=0.7023; p-Holm=1.0000; não rejeita H₀.
Internet para alunos: rural vs. urbana: p bruto=0.000158; p-Holm=0.000946; rejeita H₀.
